In [17]:
import os
import sys
from src.exception import MyException

In [3]:
%pwd

'e:\\mlops-chicken-disease-classifier\\Chicken-Disease-Classifier\\experiments'

In [4]:
os.chdir("../")

In [5]:
%pwd

'e:\\mlops-chicken-disease-classifier\\Chicken-Disease-Classifier'

In [6]:
from dataclasses import dataclass
from pathlib import Path

In [7]:
@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path    


In [8]:
from src.constants import *
from src.utils.common import read_yaml, create_directories

In [9]:
# code for configuration manager

In [ ]:
class ConfigurtionManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        return DataIngestionConfig(
            root_dir= config.root_dir,
            source_URL= config.source_URL,
            local_data_file= config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        

In [32]:
import urllib.request as request
import zipfile
from src.logger import logging
# from src.utils.common import get_size
import py7zr

In [33]:
class DataIngestion:
    def __init__(self, DataIngestionConfig):
        self.config = DataIngestionConfig

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers= request.urlretrieve(
                url = self.config.source_URL,
                filename=self.config.local_data_file
            )
            logging.info(f"{filename} download with following info: \n:{headers}")

        else:
            logging.info(f"File already exists!:{self.config.local_data_file}")

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts zip file into data directory
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with py7zr.SevenZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)

In [34]:
try:
    config = ConfigurtionManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise MyException(e, sys)

[2026-05-11 07:45:08,979] root - INFO - arctifacts/data_ingestion/data.7z download with following info: 
:Connection: close
Content-Length: 11108440
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/octet-stream
ETag: "05129671c4d983cc844ea235fe2c64439ebe122a4afb61ad8a2d4a2e2417fd16"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: ADDE:09E3:23D4D5:660537:6A013961
Accept-Ranges: bytes
Date: Mon, 11 May 2026 02:15:04 GMT
Via: 1.1 varnish
X-Served-By: cache-bom-vanm7210076-BOM
X-Cache: HIT
X-Cache-Hits: 0
X-Timer: S1778465704.127696,VS0,VE229
Vary: Authorization,Accept-Encoding
Access-Control-Allow-Origin: *
Cross-Origin-Resource-Policy: cross-origin
X-Fastly-Request-ID: 095f9a9aaf706ba13ef3677486363476db775d06
Expires: Mon, 11 May 2026 02:20:04 GMT
Source-Age: 0


